In [ ]:
# %% [markdown]
# # Region-Wise Top-20 Feature Space, Domain Diversity & Ecological Drop Analysis
#
# **Objectives**:
# 1. Inspect the types of features existing in the pipeline across remote sensing domains.
# 2. Count the unique canonical signals (keeping statistical variations intact, but aggregating to base signals to inspect redundancy).
# 3. Analyze the Top-20 features selected for each biogeographic region.
# 4. Provide a structured diagnostic and ecological explanation of features that get selected vs. those that get dropped/relegated.

# %%
from pathlib import Path
import json
import re
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Directory where ranking & uncorrelating artifacts reside
artifacts_dir = Path('data/feature_selection_artifacts')
TOP_N = 20

# 1. Load ranking artifacts
ranking_files = sorted(artifacts_dir.glob('*_ranked_features.json'))
uncorr_files = sorted(artifacts_dir.glob('*_uncorrelated_features.json'))

region_ranked = {}
for f in ranking_files:
    with open(f, 'r') as fp:
        content = json.load(fp)
        r_name = content.get('region', f.stem.replace('_ranked_features', ''))
        # Standardize ranked list whether it's dict or list of dicts/names
        raw = content.get('ranked_features', content.get('features', []))
        if isinstance(raw, dict):
            # sorted by score descending
            sorted_feats = sorted(raw.keys(), key=lambda k: raw[k], reverse=True)
            scores = raw
        elif isinstance(raw, list):
            if raw and isinstance(raw[0], dict):
                sorted_feats = [x['name'] for x in sorted(raw, key=lambda k: k.get('importance', 0), reverse=True)]
                scores = {x['name']: x.get('importance', 0) for x in raw}
            else:
                sorted_feats = raw
                scores = {name: len(raw) - idx for idx, name in enumerate(raw)}
        else:
            sorted_feats = []
            scores = {}
        
        region_ranked[r_name] = {
            'sorted_features': sorted_feats,
            'scores': scores
        }

region_uncorr = {}
for f in uncorr_files:
    with open(f, 'r') as fp:
        content = json.load(fp)
        r_name = content.get('region', f.stem.replace('_uncorrelated_features', ''))
        region_uncorr[r_name] = content.get('uncorrelated_features', [])

regions = sorted(list(region_ranked.keys()))
print(f"Loaded regions: {regions}")

# %% [markdown]
# ## 2. Feature Domain Taxonomy and Unique Canonical Token Extraction
# We distinguish between:
# - **Full Feature Name** (e.g., `ndvi_p50`, `ndvi_std`, `elevation`): Statistical reductions are kept intact.
# - **Base / Canonical Feature** (e.g., `ndvi`, `swir1`): The underlying sensor band, index, or topographic parameter.

# %%
def get_base_feature(feature_name: str) -> str:
    """Strips statistical suffixes (_p10, _p50, _p90, _mean, _std, _min, _max) to reveal the base feature."""
    stat_suffixes = [r'_p10$', r'_p25$', r'_p50$', r'_p75$', r'_p90$', r'_mean$', r'_std$', r'_variance$', r'_min$', r'_max$']
    base = feature_name
    for pattern in stat_suffixes:
        base = re.sub(pattern, '', base, flags=re.IGNORECASE)
    return base

def get_stat_suffix(feature_name: str) -> str:
    """Extracts the statistical variation/reducer attached to the feature."""
    m = re.search(r'_(p10|p25|p50|p75|p90|mean|std|variance|min|max)$', feature_name, re.IGNORECASE)
    return m.group(1).lower() if m else 'static'

def categorize_feature(feature_name: str) -> str:
    """Classifies a feature into its biophysical / remote sensing domain."""
    base = get_base_feature(feature_name).lower()
    
    topo = {'elevation', 'slope', 'aspect', 'aspect_cos', 'aspect_sin', 'hand', 'dem'}
    coords = {'lat', 'lon', 'latitude', 'longitude', 'x', 'y'}
    sma = {'gv', 'npv', 'soil', 'shade', 'ndfi'}
    bands = {'blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'tir', 'thermal'}
    indices = {'ndvi', 'evi2', 'mndwi', 'ndmi', 'bci', 'ibi', 'ndwi', 'savi'}
    tasseled_cap = {'tcb', 'tcg', 'tcw'}
    masks = {'bareearth', 'stablewater', 'transientwater', 'watermask'}
    
    if base in topo:
        return 'Topography & Hydrology'
    if base in coords:
        return 'Spatial Coordinates'
    if base in sma:
        return 'Spectral Mixture Analysis (SMA)'
    if base in indices:
        return 'Vegetation / Water Indices'
    if base in tasseled_cap:
        return 'Tasseled Cap Transformations'
    if base in bands:
        return 'Surface Reflectance Bands'
    if base in masks:
        return 'Ancillary Land Cover Masks'
    return 'Other Auxiliary'

DOMAIN_COLORS = {
    'Topography & Hydrology': '#8c564b',
    'Spatial Coordinates': '#7f7f7f',
    'Surface Reflectance Bands': '#1f77b4',
    'Vegetation / Water Indices': '#2ca02c',
    'Spectral Mixture Analysis (SMA)': '#ff7f0e',
    'Tasseled Cap Transformations': '#9467bd',
    'Ancillary Land Cover Masks': '#17becf',
    'Other Auxiliary': '#d62728'
}

# %% [markdown]
# ## 3. Feature Universe and Canonical Signal Count

# %%
# Build a comprehensive catalog of all features evaluated across all regions
feature_catalog = []
all_features_set = set()

for r in regions:
    for f in region_ranked[r]['sorted_features']:
        all_features_set.add(f)

for f in sorted(all_features_set):
    feature_catalog.append({
        'feature_name': f,
        'base_feature': get_base_feature(f),
        'temporal_stat': get_stat_suffix(f),
        'domain_category': categorize_feature(f)
    })

catalog_df = pd.DataFrame(feature_catalog)

print("="*60)
print(f"Total Unique Features in Feature Pool (Stats Included): {len(catalog_df)}")
print(f"Total Unique Base/Canonical Features (Stats Stripped): {catalog_df['base_feature'].nunique()}")
print("="*60)

# Breakdown of unique base features per category
summary_category = catalog_df.groupby('domain_category').agg(
    total_features=('feature_name', 'count'),
    unique_base_signals=('base_feature', 'nunique'),
    base_signals=('base_feature', lambda x: sorted(list(set(x))))
).reset_index()

display(summary_category)

# %% [markdown]
# ## 4. Per-Region Top-20 Selection & Composition Analysis

# %%
region_top20_records = []

for r in regions:
    top20 = region_ranked[r]['sorted_features'][:TOP_N]
    for rank, f in enumerate(top20, 1):
        region_top20_records.append({
            'region': r,
            'rank': rank,
            'feature': f,
            'base_feature': get_base_feature(f),
            'stat': get_stat_suffix(f),
            'domain': categorize_feature(f),
            'importance_score': region_ranked[r]['scores'].get(f, np.nan)
        })

top20_df = pd.DataFrame(region_top20_records)

# Visualize domain breakdown across regions
plt.figure(figsize=(14, 6))
domain_pivot = top20_df.pivot_table(
    index='region', 
    columns='domain', 
    values='feature', 
    aggfunc='count', 
    fill_value=0
)

domain_pivot.plot(kind='bar', stacked=True, figsize=(14, 6), color=[DOMAIN_COLORS.get(col, '#333333') for col in domain_pivot.columns])
plt.title(f'Domain Composition of Top-{TOP_N} Selected Features by Region', fontsize=14, fontweight='bold')
plt.xlabel('Biogeographic Region')
plt.ylabel('Number of Selected Features (out of 20)')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 5. Cross-Region Frequency & Persistence of Top-20 Features

# %%
# Unique features chosen across all regional top-20s combined
union_top20 = top20_df['feature'].unique()
union_base_top20 = top20_df['base_feature'].unique()

print(f"Total distinct features appearing in at least one regional Top-20: {len(union_top20)}")
print(f"Total distinct base signals appearing in at least one regional Top-20: {len(union_base_top20)}")

# Frequency table: How many regions pick each feature in their Top-20?
feat_freq = top20_df.groupby('feature').agg(
    region_count=('region', 'count'),
    regions=('region', lambda x: ", ".join(sorted(x))),
    domain=('domain', 'first'),
    base_signal=('base_feature', 'first')
).sort_values(by='region_count', ascending=False).reset_index()

plt.figure(figsize=(14, 5))
sns.barplot(data=feat_freq.head(20), x='feature', y='region_count', hue='domain', palette=DOMAIN_COLORS, dodge=False)
plt.title('Top 20 Most Persistently Selected Features Across Biogeographic Regions')
plt.xlabel('Feature (with statistical reducer)')
plt.ylabel('Regional Top-20 Frequency (max = 7)')
plt.xticks(rotation=45, ha='right')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 6. Systematic Regional Drop Analysis & Ecological Explanations
#
# ### Ecological Rationale Rules for Selection vs. Relegation/Drop:
# 1. **Topography (`elevation`, `slope`, `hand`)**:
#    - *Selected in `HIM`, `WGR`, `NEI`*: Critical altitudinal zonation (subtropical to alpine tree lines; orographic moisture barriers).
#    - *Dropped / low in `IGP`, `DES`*: Near-flat alluvial plains and low-relief basins provide almost zero elevation variance; spectral variability is driven by irrigation/crop cycles rather than terrain.
# 2. **Spectral Mixture Analysis (`soil`, `npv`, `gv`, `ndfi`)**:
#    - *Selected in `DES`, `ICR`, `PEN`*: Semi-arid/arid regions have open canopies, dry deciduous leaf-off periods, and exposed sub-pixel rocks/soil; unmixing decouples non-photosynthetic vegetation from background mineral substrate.
#    - *Dropped / lower rank in `NEI`, `WGR`*: Dense closed-canopy rainforests saturate bare soil fractions to near-zero year-round.
# 3. **Moisture & Water Dynamics (`mndwi`, `ndmi`, `TransientWater`, `swir2`)**:
#    - *Selected in `IGP`, `DES`, `WGR`*: Captures canal irrigation, paddy inundation cycles, reservoir drawdowns, and monsoon rain-green pulses.
#    - *Dropped / lower rank in `HIM`*: Snow and glacial ice dominate over liquid surface water dynamics; thermal and visible albedo dominate instead.
# 4. **Temporal Reducers (`_std` vs. `_p50` / `_mean`)**:
#    - When `_std` is selected, it captures **phenological amplitude** (annual crops vs. evergreen forest).
#    - When redundant percentiles (e.g., `_p50` alongside `_mean` or `_p75`) get dropped, it is due to **high collinearity ($r > 0.85$)** where median is preferred to suppress residual cloud/shadow noise.

# %%
def get_ecological_rationale(feature_name: str, region_code: str, status: str) -> str:
    """Generates biome-specific ecological rationales for selected vs. dropped features."""
    base = get_base_feature(feature_name).lower()
    stat = get_stat_suffix(feature_name).lower()
    cat = categorize_feature(feature_name)
    
    if status == 'Selected (Top-20)':
        if base in ['elevation', 'slope', 'hand']:
            return f"Selected: {region_code} has high topographic relief; elevation and slope govern biophysical thermal lapse rates, rainfall gradients, and species niches."
        elif base in ['soil', 'npv']:
            return f"Selected: In {region_code}, open canopy and dry/fallow seasons expose ground cover; sub-pixel SMA decouples dead vegetation from bare ground."
        elif base in ['swir1', 'swir2', 'ndmi']:
            return f"Selected: Strong sensitivity to canopy water stress, surface moisture, and crop irrigation dynamics in {region_code}."
        elif stat in ['std', 'variance']:
            return f"Selected: Captures temporal phenology and seasonal amplitude (e.g. crop cycle vs. natural vegetation green-up)."
        elif base in ['ndvi', 'evi2', 'tcg']:
            return f"Selected: Primary proxy for photosynthetic biomass and canopy greenness across seasonal cycles."
        else:
            return f"Selected: High discriminatory power for distinct land-cover endmembers in {region_code}."
    else: # Dropped or Relegated
        if base in ['elevation', 'slope', 'hand'] and region_code in ['IGP', 'DES']:
            return f"Dropped: Flat terrain with near-zero altitudinal variance; topography does not partition land-cover types here."
        elif base in ['soil', 'bareearth'] and region_code in ['NEI', 'WGR']:
            return f"Dropped: High-density, evergreen closed canopy leaves no bare soil exposed; signal is completely masked/saturated."
        elif base in ['blue', 'green']:
            return f"Dropped: Subordinate to NIR/SWIR; prone to atmospheric aerosol scattering and low inter-class separability."
        elif stat in ['p10', 'p25', 'p75', 'p90', 'mean']:
            return f"Dropped: Redundant proxy collinear with median (p50) or std; pruned to prevent collinear variance inflation."
        else:
            return f"Dropped: Subdued spectral variance or redundant with a higher-ranked co-spectral index in {region_code}."

# %%
# Build drop analysis table for each region
# Comparing Top-20 against features ranked 21-50 or features dropped in uncorrelation step
regional_drop_analysis = []

for r in regions:
    top20_set = set(region_ranked[r]['sorted_features'][:TOP_N])
    # Consider the next tier of features or those dropped
    all_ranked = region_ranked[r]['sorted_features']
    relegated_features = all_ranked[TOP_N:TOP_N + 15] # Features ranked 21-35
    
    # 1. Top 20 Entries
    for rank, feat in enumerate(all_ranked[:TOP_N], 1):
        regional_drop_analysis.append({
            'Region': r,
            'Rank': rank,
            'Feature': feat,
            'Base_Feature': get_base_feature(feat),
            'Domain': categorize_feature(feat),
            'Status': 'Selected (Top-20)',
            'Ecological_Rationale': get_ecological_rationale(feat, r, 'Selected (Top-20)')
        })
        
    # 2. Notable Relegated / Dropped Entries
    for rank, feat in enumerate(relegated_features, TOP_N + 1):
        regional_drop_analysis.append({
            'Region': r,
            'Rank': rank,
            'Feature': feat,
            'Base_Feature': get_base_feature(feat),
            'Domain': categorize_feature(feat),
            'Status': 'Relegated / Dropped',
            'Ecological_Rationale': get_ecological_rationale(feat, r, 'Dropped')
        })

analysis_df = pd.DataFrame(regional_drop_analysis)

# %% [markdown]
# ### Region Diagnostic Sample: Inspecting Selection vs. Drop Rationale for a Given Biome

# %%
# Example: Inspecting Desert (DES) vs Western Ghats (WGR) vs Indo-Gangetic Plains (IGP)
sample_region = 'WGR'
print(f"=== Feature Selection and Drop Analysis for Region: {sample_region} ===")
display(analysis_df[analysis_df['Region'] == sample_region][['Rank', 'Feature', 'Domain', 'Status', 'Ecological_Rationale']].head(25))

# %% [markdown]
# ## 7. Summary Matrix of Redundant Suffixes vs. Retained Signals
# Evaluates whether statistical variants (`_p50` vs `_std` vs `_mean`) are co-selected in the Top 20.

# %%
stat_pivot = top20_df.pivot_table(
    index='base_feature', 
    columns='stat', 
    values='feature', 
    aggfunc='count', 
    fill_value=0
)

plt.figure(figsize=(10, 8))
sns.heatmap(stat_pivot, annot=True, fmt='d', cmap='YlGnBu', cbar_kws={'label': 'Selection Count Across Regional Top-20s'})
plt.title('Distribution of Statistical Reducers per Base Feature in Regional Top-20s')
plt.xlabel('Statistical Variation / Reducer')
plt.ylabel('Base Feature Signal')
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 8. Exporting Summary Tables to CSV for Reporting

# %%
output_csv_path = Path('top20_feature_space_and_drop_analysis.csv')
analysis_df.to_csv(output_csv_path, index=False)
print(f"Detailed drop analysis successfully exported to: {output_csv_path.resolve()}")